# 02 - Task Similarity Matrix

This notebook expands the workflow from `examples/compute_task_similarity.py`:

1. build a deterministic multitask model;
2. compute ALE profiles;
3. compare task curves with the Frechet-based similarity helper;
4. inspect nearest-task groups.

In [1]:
from pathlib import Path
import sys

repo_root = next(
    (
        path
        for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
        if (path / "pyproject.toml").exists() and (path / "src" / "alemtl").exists()
    ),
    Path.cwd().resolve().parent if Path.cwd().resolve().name == "notebooks" else Path.cwd().resolve(),
)
for path in (repo_root, repo_root / "src"):
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

import torch

from examples.compute_ale_profiles import build_model, make_batches
from alemtl.similarity import MultiTaskALE, MultitaskSimilarity

torch.manual_seed(5)

## ALE Profiles

The deterministic model has three task-specific linear heads. Tasks 0 and 1 are deliberately similar; task 2 is deliberately different.

In [2]:
model = build_model()
batches = make_batches(n_batches=8, batch_size=48)

ale = MultiTaskALE(
    model=model,
    dataloader=batches,
    n_tasks=3,
    n_features_out=1,
    num_intervals=12,
    n_guess=96,
)
ale.update()
curves = ale(centered=True, cumulative=True, std=1.0)
curves.shape

torch.Size([3, 2, 12, 2])

Curve layout is `(tasks, features, intervals, x_plus_outputs)`. With one output, the last axis is `(x, y)`.

In [3]:
task = 0
feature = 0
curves[task, feature, :5]

tensor([[-1.7617, -1.7693],
        [-0.9048, -0.9838],
        [-0.6633, -0.7624],
        [-0.4483, -0.5653],
        [-0.1057, -0.2513]])

## Pairwise Task Similarity

`MultitaskSimilarity.scores` stores task similarity scores, where larger means more similar.

In [4]:
similarity = MultitaskSimilarity(ale_curves=ale, centered=True, cumulative=True, std=1.0)
similarity.compute()
similarity.scores

tensor([[0.0000, 0.5603, 0.0232],
        [0.5603, 0.0000, 0.0250],
        [0.0232, 0.0250, 0.0000]])

## Feature-level Similarity

The feature tensor keeps one score per `(task, task, feature)`.

In [5]:
similarity.similarity_tasks_features.shape, similarity.similarity_tasks_features

(torch.Size([3, 3, 2]),
 tensor([[[0.0000, 0.0000],
          [0.4723, 0.0880],
          [0.0169, 0.0063]],
 
         [[0.4723, 0.0880],
          [0.0000, 0.0000],
          [0.0146, 0.0104]],
 
         [[0.0169, 0.0063],
          [0.0146, 0.0104],
          [0.0000, 0.0000]]]))

## Nearest-task Groups

These pairs are what the loss regularizer can consume through `update_tasks_groups`.

In [6]:
scores, pairs = similarity.tasks_groups()
for score, pair in zip(scores, pairs):
    task, nearest = pair.tolist()
    print(f"task {task} -> task {nearest}, score={score.item():.4f}")

task 0 -> task 1, score=0.5603
task 1 -> task 0, score=0.5603
task 2 -> task 1, score=0.0250
